# 04 - Modeling and Evaluation
Covers Deliverable D: Baselines, ML Models, K-Fold CV, Hyperparameter Tuning, Feature Importance, and Model Saving.


In [ ]:
import os, sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.model_selection import train_test_split, KFold, cross_validate, GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.inspection import permutation_importance
import joblib

# DYNAMIC PATH RESOLUTION
def resolve_paths():
    for p in ['../data/raw', 'data/raw', 'team_6/data/raw', '../../data/raw']:
        if os.path.exists(os.path.join(p, 'crop_yield_train.csv')):
            raw = p
            base = os.path.dirname(p)
            proc = os.path.join(base, 'processed')
            fig = os.path.join(os.path.dirname(base), 'figures')
            mod = os.path.join(os.path.dirname(base), 'models')
            return raw, proc, fig, mod
    return '../data/raw', '../data/processed', '../figures', '../models'

RAW_DIR, PROCESSED_DIR, FIGURES_DIR, MODELS_DIR = resolve_paths()
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

print(f"PASS: Located processed data at '{PROCESSED_DIR}'.")


## D1 & D2: Data Preparation & Preprocessing Pipeline


In [ ]:
# Load master dataset
train_df = pd.read_csv(os.path.join(PROCESSED_DIR, 'master_train.csv'))

# Features & Target split
X = train_df.drop(columns=['plot_id', 'yield_tons_per_ha'], errors='ignore')
y = train_df['yield_tons_per_ha']

# Identify numeric vs categorical columns
numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

print(f"Features count: {X.shape[1]} (Numeric: {len(numeric_cols)}, Categorical: {len(categorical_cols)})")

# Build Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', SimpleImputer(strategy='median'), numeric_cols),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), categorical_cols)
    ]
)
print("Preprocessor pipeline configured successfully.")


## D3: Model Comparison & 5-Fold Cross-Validation


In [ ]:
# Train-Validation Split for holdout evaluation
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Define models list (Baselines, Linear, and Tree Ensembles)
models = {
    'Dummy Mean Baseline': DummyRegressor(strategy='mean'),
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0, random_state=42),
    'Hist Gradient Boosting': HistGradientBoostingRegressor(max_iter=50, random_state=42),
    'Random Forest Regressor': RandomForestRegressor(n_estimators=20, max_depth=12, random_state=42)
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)
results = []

print("Running 5-Fold Cross-Validation across model candidates...")
for name, model in models.items():
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    
    cv_scores = cross_validate(
        pipe, X, y, cv=kf,
        scoring={'rmse': 'neg_root_mean_squared_error', 'mae': 'neg_mean_absolute_error', 'r2': 'r2'}
    )
    
    rmse_cv = -cv_scores['test_rmse'].mean()
    mae_cv = -cv_scores['test_mae'].mean()
    r2_cv = cv_scores['test_r2'].mean()
    
    results.append({
        'Model': name,
        'CV RMSE': rmse_cv,
        'CV MAE': mae_cv,
        'CV R2': r2_cv
    })
    print(f"  --> {name}: CV RMSE = {rmse_cv:.4f} | CV MAE = {mae_cv:.4f} | CV R2 = {r2_cv:.4f}")

results_df = pd.DataFrame(results).sort_values(by='CV RMSE')
print("--- Cross Validation Summary ---")
print(results_df.to_string(index=False))


## D4: Hyperparameter Tuning (GridSearchCV)


In [ ]:
# Hyperparameter tuning for top performing model
hgb_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', HistGradientBoostingRegressor(random_state=42))
])

param_grid = {
    'model__max_iter': [50, 100],
    'model__max_depth': [10, 15]
}

print("Running GridSearchCV for HistGradientBoostingRegressor...")
grid_search = GridSearchCV(
    hgb_pipe, param_grid, cv=3,
    scoring='neg_root_mean_squared_error'
)
grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_
print("Best Hyperparameters:", grid_search.best_params_)

# Evaluate best model on Validation Holdout
val_preds = best_model.predict(X_val)
final_rmse = np.sqrt(mean_squared_error(y_val, val_preds))
final_mae = mean_absolute_error(y_val, val_preds)
final_r2 = r2_score(y_val, val_preds)

print("Final Validation Metrics (Best Model):")
print(f"  - RMSE : {final_rmse:.4f}")
print(f"  - MAE  : {final_mae:.4f}")
print(f"  - R2   : {final_r2:.4f}")


## D5: Permutation Feature Importance & Figure 12 Generation


In [ ]:
# Fit best model on entire dataset for production export
best_model.fit(X, y)

# Permutation Feature Importance
perm_importance = permutation_importance(
    best_model, X_val, y_val, n_repeats=2, random_state=42
)

feature_names = X.columns
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': perm_importance.importances_mean
}).sort_values(by='Importance', ascending=False)

print("--- Top 10 Features by Permutation Importance ---")
print(importance_df.head(10).to_string(index=False))

# Plot Figure 12
plt.figure(figsize=(9, 5.5))
sns.barplot(
    data=importance_df.head(10),
    x='Importance', y='Feature',
    hue='Feature', palette='crest', legend=False
)
plt.title("Figure 12: Permutation Feature Importance (Top 10 Drivers)", fontsize=12, fontweight='bold', pad=12)
plt.xlabel("Decrease in Model Score (Importance)", fontsize=10)
plt.ylabel("Feature", fontsize=10)
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()

fig12_path = os.path.join(FIGURES_DIR, 'fig12_feature_importance.png')
plt.savefig(fig12_path, dpi=300)
plt.close()
print(f"Saved Figure 12 plot to '{fig12_path}'.")


## D6: Export Pipeline Model (Deliverable E Readiness)


In [ ]:
# Save full scikit-learn Pipeline (Preprocessing + Estimator)
model_export_path = os.path.join(MODELS_DIR, 'final_model.joblib')
joblib.dump(best_model, model_export_path)

print(f"SUCCESS: Exported full pipeline model to '{model_export_path}'!")
print("Model ready for Streamlit demo app inference!")
